# Extract TikTok captions and subtitles

This notebook produces one JSON file. It does not download MP4 videos or save separate VTT files. Use only public videos that you own or have permission to process.

In [4]:
%pip install -q -U yt-dlp


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import re
import tempfile
from pathlib import Path

from yt_dlp import YoutubeDL

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add public TikTok URLs here.
VIDEO_URLS = [
    "https://www.tiktok.com/@haylawtalk/video/7572833218043399442?is_from_webapp=1&sender_device=pc"
]

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'judement' / 'tiktok'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = OUTPUT_DIR / 'tiktok_captions.json'

def vtt_to_raw_text(vtt_text: str) -> str:
    khmer_segments = []
    previous = None
    for raw_line in vtt_text.splitlines():
        line = raw_line.strip()
        if not line or line == 'WEBVTT' or line.startswith('NOTE'):
            continue
        if '-->' in line or re.fullmatch(r'\d+', line):
            continue
        line = re.sub(r'<[^>]+>', '', line)
        # Keep Khmer letters, Khmer signs/symbols, and whitespace only.
        line = re.sub(r'[^\u1780-\u17FF\u19E0-\u19FF\s]', ' ', line)
        line = ' '.join(line.split())
        if line and re.search(r'[\u1780-\u17FF]', line) and line != previous:
            khmer_segments.append(line)
            previous = line
    return ' '.join(khmer_segments)

def find_subtitle_file(directory: Path, video_id: str) -> Path | None:
    files = sorted(directory.glob(f'{video_id}.*.vtt'))
    if not files:
        return None
    preferred = [file for file in files if any(language in file.name for language in ('khm-KH', 'kh', 'km'))]
    return preferred[0] if preferred else files[0]

records = []
with tempfile.TemporaryDirectory(prefix='tiktok-captions-') as temporary_dir:
    temporary_path = Path(temporary_dir)
    options = {
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': ['all'],
        'subtitlesformat': 'vtt/best',
        'outtmpl': str(temporary_path / '%(id)s.%(ext)s'),
        'noplaylist': True,
        'ignoreerrors': True,
        'quiet': False,
    }
    with YoutubeDL(options) as downloader:
        for url in VIDEO_URLS:
            try:
                info = downloader.extract_info(url, download=True)
                if not info:
                    records.append({'url': url, 'status': 'not_found'})
                    continue
                video_id = str(info.get('id', ''))
                subtitle_file = find_subtitle_file(temporary_path, video_id)
                vtt_text = subtitle_file.read_text(encoding='utf-8', errors='replace') if subtitle_file else ''
                records.append({
                    'video_id': video_id,
                    'url': info.get('webpage_url') or url,
                    'uploader': info.get('uploader'),
                    'post_caption': info.get('description') or '',
                    'subtitle_language': subtitle_file.name.split('.')[1] if subtitle_file else None,
                    'subtitle_vtt': vtt_text,
                    'subtitle_text': vtt_to_raw_text(vtt_text) if vtt_text else '',
                    'status': 'ok' if vtt_text else 'no_subtitle_found',
                })
            except Exception as error:
                records.append({'url': url, 'status': 'error', 'error': str(error)})

OUTPUT_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Saved {len(records)} records to {OUTPUT_JSON}')

[TikTok] Extracting URL: https://www.tiktok.com/@haylawtalk/video/7572833218043399442?is_from_webapp=1&sender_device=pc
[TikTok] 7572833218043399442: Downloading webpage


[info] 7572833218043399442: Downloading subtitles: eng-US, khm-KH
[info] Testing format bytevc1_720p_298131-1
[info] 7572833218043399442: Downloading 1 format(s): bytevc1_720p_298131-1+audio
[info] Writing video subtitles to: /var/folders/vc/zbr37wk97sj_74bcsp7gmbr00000gn/T/tiktok-captions-tleohwnw/7572833218043399442.eng-US.vtt
[download] Destination: /var/folders/vc/zbr37wk97sj_74bcsp7gmbr00000gn/T/tiktok-captions-tleohwnw/7572833218043399442.eng-US.vtt
[download] 100% of    1.83KiB in 00:00:00 at 22.74KiB/s    
[info] Writing video subtitles to: /var/folders/vc/zbr37wk97sj_74bcsp7gmbr00000gn/T/tiktok-captions-tleohwnw/7572833218043399442.khm-KH.vtt
[download] Destination: /var/folders/vc/zbr37wk97sj_74bcsp7gmbr00000gn/T/tiktok-captions-tleohwnw/7572833218043399442.khm-KH.vtt
[download] 100% of    3.58KiB in 00:00:00 at 159.76KiB/s   
Saved 1 records to /Users/nak/CADT/projects/legal-rag/data/judement/tiktok/tiktok_captions.json
